# Flight Fare Prediction — Assignment 1

## 1. Problem definition
Estimate the listed Indian flight fare (`price`) from itinerary and booking information. This is a supervised regression problem. RMSE is the primary metric; MAE and R² provide complementary views. A useful model should beat a median-fare baseline on the untouched test set; R² ≥ 0.80 is a stretch goal, not an assumed outcome.

This notebook uses the downloaded CSV locally and does not fetch external data. Run it from this project directory after installing dependencies with `uv sync`.

In [2]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

search_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in search_roots if (path / "src").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    PROJECT_ROOT = next(
        (
            path / "flight-price-prediction"
            for path in search_roots
            if (path / "flight-price-prediction" / "src").is_dir()
        ),
        None,
    )
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project directory containing src/")

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from flight_price_prediction.pipeline import (
    clean_flight_data,
    load_dataset,
    make_features,
    predict_fares,
    run_experiment,
)

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Clean_Dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
sns.set_theme(style="whitegrid")

search_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in search_roots if (path / "src").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    PROJECT_ROOT = next(
        (
            path / "flight-price-prediction"
            for path in search_roots
            if (path / "flight-price-prediction" / "src").is_dir()
        ),
        None,
    )
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project directory containing src/")

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)


DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Clean_Dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
sns.set_theme(style="whitegrid")

## 2–3. Data collection, exploration, and preparation
The source file is the locally downloaded Kaggle dataset. Start with its dimensions, schema, missingness, duplicate count, and target distribution before training.

In [3]:
raw = load_dataset(DATA_PATH)
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
display(raw.head())
display(raw.dtypes.rename("dtype").to_frame())
display(raw.isna().sum().rename("missing_values").to_frame())
print(f"Exact duplicate rows: {raw.duplicated().sum():,}")
display(raw["price"].describe().to_frame().T)

Rows: 300,153 | Columns: 12


,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


,dtype
Unnamed: 0,int64
airline,str
flight,str
source_city,str
departure_time,str
stops,str
arrival_time,str
destination_city,str
class,str
duration,float64


,missing_values
Unnamed: 0,0
airline,0
flight,0
source_city,0
departure_time,0
stops,0
arrival_time,0
destination_city,0
class,0
duration,0


Exact duplicate rows: 0


,count,mean,std,min,25%,50%,75%,max
price,300153.0,20889.660523,22697.767366,1105.0,4783.0,7425.0,42521.0,123071.0


### Explore distributions and relationships
Fares are typically right-skewed. The log-scaled x-axis makes the lower-fare range visible. Class and airline comparisons are descriptive, not causal. Premium-class fares can be legitimate high values, so the workflow does not remove fare outliers simply because they are large.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=raw, x="price", bins=60, log_scale=(True, False), ax=axes[0])
axes[0].set(title="Fare distribution (log-scaled x-axis)", xlabel="Fare")
sns.boxplot(data=raw, x="class", y="price", showfliers=False, ax=axes[1])
axes[1].set(title="Fare by class (outliers hidden for readability)", ylabel="Fare")
plt.tight_layout()
plt.show()

display(
    raw.groupby("airline", observed=True)["price"]
    .median()
    .sort_values()
    .rename("median_fare")
    .to_frame()
)
display(raw.select_dtypes(include="number").corr().round(2))

### Clean and engineer features
Remove exact duplicates, the exported CSV row index, and rows without a valid target. Keep feature gaps for imputation inside the training pipeline. The model uses itinerary, class, duration, and days-to-departure; it adds a source-to-destination route. The flight code is excluded because its many values can encourage memorization.

In [ ]:
cleaned, cleaning_report = clean_flight_data(raw)
features = make_features(cleaned)
target = cleaned["price"]
print("Cleaning summary:", cleaning_report)
print(f"Prepared feature shape: {features.shape}; target shape: {target.shape}")
display(features.head())

## 4–6. Model development, validation, and tuning
The reusable workflow splits data into 80% train, 10% validation, and 10% test (random seed 42). Numeric values are median-imputed and scaled; categories are mode-imputed and one-hot encoded. It compares a median baseline and Ridge regression with a HistGradientBoosting regressor tuned by randomized search and 3-fold cross-validation on training data only. Validation RMSE selects the final model. The test set is used only for the final generalization check.

In [ ]:
results = run_experiment(
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    search_iterations=4,
    cv_folds=3,
)
print("Selected model:", results["selected_model"])
print("Split sizes:", results["split_rows"])
display(pd.DataFrame(results["validation_metrics"]).T.sort_values("rmse"))

## 7. Final test evaluation
Lower RMSE and MAE are better; higher R² is better. RMSE is more sensitive to large fare errors than MAE. Compare with the median baseline and the stated success check. Report the measured values—do not claim a target was achieved unless the output supports it.

In [ ]:
test_results = pd.DataFrame(
    {
        "selected_model": results["test_metrics"],
        "median_baseline": results["median_baseline_test_metrics"],
    }
).T
display(test_results)
print(
    "Beats median baseline by test RMSE:",
    results["test_metrics"]["rmse"] < results["median_baseline_test_metrics"]["rmse"],
)
print("Stretch goal R² ≥ 0.80:", results["test_metrics"]["r2"] >= 0.80)
print("Saved report:", OUTPUT_DIR / "reports" / "metrics.json")

## 8–10. Local inference demo and maintenance
The fitted preprocessing and regression steps are saved together, so new records receive the same transformations. This is a local deployment demonstration; a hosted production service would additionally need request validation, monitoring, and retraining procedures.

In [ ]:
example_flight = {
    "airline": "Vistara",
    "source_city": "Delhi",
    "departure_time": "Morning",
    "stops": "one",
    "arrival_time": "Afternoon",
    "destination_city": "Mumbai",
    "class": "Economy",
    "duration": 2.5,
    "days_left": 10,
}
model_path = OUTPUT_DIR / "models" / "flight_price_model.joblib"
print("Predicted fare:", predict_fares(model_path, [example_flight])[0])

## 11. Documentation and presentation
Use the saved plots and metric tables in the presentation. Explain the business question, data cleaning, feature choices, validation-based model selection, test results, and limitations (including that listed historical fares may not represent current market prices). Discuss what you would monitor and how you would respond to drift. Add your own interpretation of the results from this run.